In [1]:
import socket
import threading
import os


In [ ]:
clients = []
names = {}

ALLOWED_EXTENSIONS = [".txt", ".jpg", ".pdf"]
BANNED_WORDS = ["badword", "hello"]

In [3]:
def remove_client(c):
    if c in clients:
        clients.remove(c)
    if c in names:
        del names[c]
    c.close()

In [4]:
def broadcast(message, sender=None):
    for client in clients:
        if client != sender:
            try:
                client.send(message)
            except:
                remove_client(client)

In [5]:
def handle_client(c):
    while True:
        try:
            msg = c.recv(1024).decode()

            if msg == "":
                break

            if msg.lower() == "exit":
                name = names[c]
                print(name, "left the chat")

                for client in clients:
                    if client != c:
                        client.send((name + " left the chat").encode())

                clients.remove(c)
                del names[c]
                c.close()
                break

            # File handling
            if msg.startswith("FILE:"):
                parts = msg.split(":")
                filename = parts[1]
                filesize = int(parts[2])

                ext = os.path.splitext(filename)[1]

                if ext not in ALLOWED_EXTENSIONS:
                    c.send("SERVER: File type not allowed!".encode())
                    continue

                f = open("server_" + filename, "wb")
                remaining = filesize

                while remaining > 0:
                    data = c.recv(1024)
                    f.write(data)
                    remaining = remaining - len(data)

                f.close()

                for client in clients:
                    if client != c:
                        client.send(("FILE:" + filename + ":" + str(filesize)).encode())

                        f = open("server_" + filename, "rb")
                        while True:
                            data = f.read(1024)
                            if not data:
                                break
                            client.send(data)
                        f.close()

                continue

            # Banned words check
            bad = False
            for word in BANNED_WORDS:
                if word in msg.lower():
                    bad = True
                    break

            if bad:
                c.send("SERVER: Message contains banned words!".encode())
            else:
                for client in clients:
                    if client != c:
                        client.send(msg.encode())

        except:
            if c in clients:
                clients.remove(c)
            c.close()
            break


In [6]:
s = socket.socket()
s.bind(('localhost', 9999))
s.listen(5)

print("Server started...")
print("Waiting for connections...")

Server started...
Waiting for connections...


In [ ]:
while True:
    c, addr = s.accept()

    name = c.recv(1024).decode()
    clients.append(c)
    names[c] = name

    print(f"{name} connected")

    broadcast(f"{name} joined the chat".encode())

    threading.Thread(target=handle_client, args=(c,), daemon=True).start()

rayyan connected
rayyan connected
malik connected
arham connected
